In [1]:
import numpy as np

# --- 1. SETUP PROBLEM CONSTANTS ---
NUM_CITIES = 5          # Keep it small to clearly observe convergence
NUM_ANTS = 5            # Number of ants exploring per iteration
NUM_ITERATIONS = 50     # How many times the colony searches
ALPHA = 1.0             # Pheromone importance factor
BETA = 2.0              # Distance/Heuristic importance factor
EVAPORATION_RATE = 0.5  # Evaporation rate (rho)
Q = 100                 # Pheromone deposit intensity factor

# Generate static random coordinates for delivery cities (Depot is City 0)
np.random.seed(42)
city_coordinates = np.random.rand(NUM_CITIES, 2) * 100  # Cities placed in a 100x100 grid

# Build the Distance Matrix
distance_matrix = np.zeros((NUM_CITIES, NUM_CITIES))
for i in range(NUM_CITIES):
    for j in range(NUM_CITIES):
        if i != j:
            distance_matrix[i][j] = np.linalg.norm(city_coordinates[i] - city_coordinates[j])
        else:
            distance_matrix[i][j] = np.inf  # Prevent moving to the same city

# Initialize pheromone matrix uniformly
pheromone_matrix = np.ones((NUM_CITIES, NUM_CITIES))

# --- 2. ACO ALGORITHM RUN ---
best_distance = np.inf
best_route = []

print("Starting Ant Colony Optimization for Route Optimization...\n")

for iteration in range(NUM_ITERATIONS):
    all_ant_routes = []
    all_ant_distances = []

    for ant in range(NUM_ANTS):
        # Every ant starts at the Depot (City 0)
        current_city = 0
        visited = [current_city]
        total_distance = 0.0

        # Step through all remaining cities
        while len(visited) < NUM_CITIES:
            probabilities = []
            unvisited_cities = [c for c in range(NUM_CITIES) if c not in visited]

            # Calculate selection values for each unvisited destination
            denominator = 0.0
            for unvisited in unvisited_cities:
                pheromone = pheromone_matrix[current_city][unvisited] ** ALPHA
                heuristic = (1.0 / distance_matrix[current_city][unvisited]) ** BETA
                denominator += pheromone * heuristic

            # Calculate exact probability for each path choice
            for unvisited in unvisited_cities:
                pheromone = pheromone_matrix[current_city][unvisited] ** ALPHA
                heuristic = (1.0 / distance_matrix[current_city][unvisited]) ** BETA
                prob = (pheromone * heuristic) / denominator
                probabilities.append(prob)

            # Probabilistically select the next city
            next_city = np.random.choice(unvisited_cities, p=probabilities)

            # Update path trackers
            total_distance += distance_matrix[current_city][next_city]
            visited.append(next_city)
            current_city = next_city

        # Return to the starting depot (City 0)
        total_distance += distance_matrix[current_city][0]
        visited.append(0)

        all_ant_routes.append(visited)
        all_ant_distances.append(total_distance)

        # Keep track of global absolute best solution found
        if total_distance < best_distance:
            best_distance = total_distance
            best_route = visited

    # --- 3. PHEROMONE UPDATE STAGE ---
    # Evaporate current pheromones
    pheromone_matrix *= (1.0 - EVAPORATION_RATE)

    # Deposit new pheromones based on the quality of ant tours
    for ant_idx in range(NUM_ANTS):
        route = all_ant_routes[ant_idx]
        dist = all_ant_distances[ant_idx]
        pheromone_to_add = Q / dist

        # Apply pheromone to each leg of that ant's journey
        for step in range(NUM_CITIES):
            from_city = route[step]
            to_city = route[step + 1]
            pheromone_matrix[from_city][to_city] += pheromone_to_add
            pheromone_matrix[to_city][from_city] += pheromone_to_add  # Symmetric path

    # Optional progress output printout
    if (iteration + 1) % 10 == 0 or iteration == 0:
        print(f"Iteration {iteration + 1}/{NUM_ITERATIONS} | Best Distance So Far: {best_distance:.2f} km")

# --- 4. OUTPUT RESULTS ---
print("\n" + "="*40)
print("OPTIMIZATION COMPLETE")
print("="*40)
print(f"Shortest Delivery Route Found: {' -> '.join(map(str, best_route))}")
print(f"Minimum Travel Distance:      {best_distance:.2f} km")


Starting Ant Colony Optimization for Route Optimization...

Iteration 1/50 | Best Distance So Far: 227.35 km
Iteration 10/50 | Best Distance So Far: 227.35 km
Iteration 20/50 | Best Distance So Far: 227.35 km
Iteration 30/50 | Best Distance So Far: 227.35 km
Iteration 40/50 | Best Distance So Far: 227.35 km
Iteration 50/50 | Best Distance So Far: 227.35 km

OPTIMIZATION COMPLETE
Shortest Delivery Route Found: 0 -> 3 -> 2 -> 1 -> 4 -> 0
Minimum Travel Distance:      227.35 km
